In [89]:
import pandas as pd
import os
from pathlib import Path

Loading the datasets

In [90]:
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath("..")
data_folder = Path(parent_dir, "fetched data")

# economic_indicators = ["gdp", "gdp_per_capita", "unemployment_rate"]
gdp_df = pd.read_csv(data_folder / "gdp.csv")
gdp_pc_df = pd.read_csv(data_folder / "gdp_per_capita.csv")
unemployment_df = pd.read_csv(data_folder / "unemployment_rate.csv")

# health_indicators
population_df = pd.read_csv(data_folder / "population.csv")
life_expectancy_df = pd.read_csv(data_folder / "life_expectancy.csv")
youth_mortality_df = pd.read_csv(data_folder / "youth_mortality.csv")
homicides_df = pd.read_csv(data_folder / "homicide_rate.csv")

# social_indicators
literacy_df = pd.read_csv(data_folder / "literacy_rate.csv")
human_rights_df = pd.read_csv(data_folder / "human_rights.csv")

# poverty_indicators
share_poverty_df = pd.read_csv(data_folder / "share_in_poverty.csv")
number_poverty_df = pd.read_csv(data_folder / "number_in_poverty.csv")

merging datasets based on the relatablity

In [91]:
economic_indicators_df = pd.merge(gdp_df, gdp_pc_df, on=["Entity", "Year"], how="outer").merge(unemployment_df, on=["Entity", "Year"], how="outer")
health_indicators_df = pd.merge(population_df, life_expectancy_df, left_on=["entity", "year"], right_on=["Entity", "Year"], how="outer").merge(youth_mortality_df, on=["Entity", "Year"], how="outer").merge(homicides_df, on=["Entity", "Year"], how="outer")
social_indicators_df = pd.merge(literacy_df, human_rights_df, on=["Entity", "Year"], how="outer")
poverty_indicators_df = pd.merge(share_poverty_df, number_poverty_df, on=["Country", "Year"], how="outer")


giving aliases to merged datasets

In [92]:
ei = economic_indicators_df
hi = health_indicators_df
si = social_indicators_df
pi = poverty_indicators_df

checking the max years in all datasets to get the mutual and latest year in all of them

In [93]:
list_of_data = [ei[ei.Year == ei.Year.max()].Year.unique(), 
                hi[hi.Year == hi.Year.max()].Year.unique(), 
                si[si.Year == si.Year.max()].Year.unique(),
                pi[pi.Year == pi.Year.max()].Year.unique()]

print(f"Latest common year in all datasets: {min(list_of_data)}")

Latest common year in all datasets: [2024.]


Renaming & Selecting (changing column names, selecting useful ones)

In [94]:
# economic indicators dataset
ei.rename({'Entity': 'country',
           'Year': 'year',
           'GDP': 'gdp',
           'GDP per capita': 'gdp_pc',
           'World region according to OWID': 'region',
           'Unemployment rate': 'unemployment_rate'}, axis=1, inplace=True)
ei = ei[['country', 'region', 'year', 'gdp', 'gdp_pc', 'unemployment_rate']]


# health indicators dataset
hi.rename({'entity': 'country',
           'population__sex_all__age_all__variant_estimates': 'total_population',
           'Life expectancy': 'life_expectancy',
           'Under-fifteen mortality rate': 'mortality_rate_u15',
           'Homicide rate per 100,000 population': 'homicide_rate_p100k',
           'World region according to OWID': 'region'}, axis=1, inplace=True)
hi = hi[['country', 'region', 'year', 'total_population', 'life_expectancy', 'mortality_rate_u15', 'homicide_rate_p100k']]


# social indicators dataset
si.rename({'Entity': 'country',
           'Year': 'year',
           'Literacy rate': 'literacy_rate',
           'Human Rights Index': 'human_rights_index',
           'World region according to OWID': 'region'}, axis=1, inplace=True)
si = si[['country', 'region', 'year', 'literacy_rate', 'human_rights_index']]



# poverty_indicators_df
pi.rename({'Country': 'country',
           'Year': 'year',
           'Share below $3 a day': 'poverty_share',
           'Number below $3 a day': 'poverty_number'}, axis=1, inplace=True)
pi = pi[['country', 'year', 'poverty_share', 'poverty_number']]



Selecting the data only between the span of 10 years from 2015 - 2024

In [95]:
ei = ei[(ei.year > 2014) & (ei.year < 2025)]
hi = hi[(hi.year > 2014) & (hi.year < 2025)]
si = si[(si.year > 2014) & (si.year < 2025)]
pi = pi[(pi.year > 2014) & (pi.year < 2025)]

choosing the standard UN-recognized countries and dropping the other entities like Central Asia, Rural China, and such

In [96]:
countries_to_keep = [
    "Afghanistan", "Albania", "Algeria", "Andorra", "Angola", "Antigua and Barbuda", 
    "Argentina", "Armenia", "Australia", "Austria", "Azerbaijan", "Bahamas", 
    "Bahrain", "Bangladesh", "Barbados", "Belarus", "Belgium", "Belize", 
    "Benin", "Bhutan", "Bolivia", "Bosnia and Herzegovina", "Botswana", "Brazil", 
    "Brunei", "Bulgaria", "Burkina Faso", "Burundi", "Cambodia", "Cameroon", 
    "Canada", "Cape Verde", "Central African Republic", "Chad", "Chile", 
    "China", "Colombia", "Comoros", "Congo", "Costa Rica", "Cote d'Ivoire", 
    "Croatia", "Cuba", "Cyprus", "Czechia", "Democratic Republic of Congo", 
    "Denmark", "Djibouti", "Dominica", "Dominican Republic", "East Timor", 
    "Ecuador", "Egypt", "El Salvador", "Equatorial Guinea", "Eritrea", "Estonia", 
    "Eswatini", "Ethiopia", "Fiji", "Finland", "France", "Gabon", "Gambia", 
    "Georgia", "Germany", "Ghana", "Greece", "Grenada", "Guatemala", "Guinea", 
    "Guinea-Bissau", "Guyana", "Haiti", "Honduras", "Hungary", "Iceland", 
    "India", "Indonesia", "Iran", "Iraq", "Ireland", "Israel", "Italy", 
    "Jamaica", "Japan", "Jordan", "Kazakhstan", "Kenya", "Kiribati", "Kosovo", 
    "Kuwait", "Kyrgyzstan", "Laos", "Latvia", "Lebanon", "Lesotho", "Liberia", 
    "Libya", "Liechtenstein", "Lithuania", "Luxembourg", "Madagascar", "Malawi", 
    "Malaysia", "Maldives", "Mali", "Malta", "Marshall Islands", "Mauritania", 
    "Mauritius", "Mexico", "Micronesia (country)", "Moldova", "Monaco", 
    "Mongolia", "Montenegro", "Morocco", "Mozambique", "Myanmar", "Namibia", 
    "Nauru", "Nepal", "Netherlands", "New Zealand", "Nicaragua", "Niger", 
    "Nigeria", "North Korea", "North Macedonia", "Norway", "Oman", "Pakistan", 
    "Palau", "Palestine", "Panama", "Papua New Guinea", "Paraguay", "Peru", 
    "Philippines", "Poland", "Portugal", "Qatar", "Romania", "Russia", 
    "Rwanda", "Saint Kitts and Nevis", "Saint Lucia", "Saint Vincent and the Grenadines", 
    "Samoa", "San Marino", "Sao Tome and Principe", "Saudi Arabia", "Senegal", 
    "Serbia", "Seychelles", "Sierra Leone", "Singapore", "Slovakia", "Slovenia", 
    "Solomon Islands", "Somalia", "South Africa", "South Korea", "South Sudan", 
    "Spain", "Sri Lanka", "Sudan", "Suriname", "Sweden", "Switzerland", "Syria", 
    "Tajikistan", "Tanzania", "Thailand", "Togo", "Tonga", "Trinidad and Tobago", 
    "Tunisia", "Turkey", "Turkmenistan", "Tuvalu", "Uganda", "Ukraine", 
    "United Arab Emirates", "United Kingdom", "United States", "Uruguay", 
    "Uzbekistan", "Vanuatu", "Venezuela", "Vietnam", "Yemen", "Zambia", "Zimbabwe"
]

len(countries_to_keep)  # now this list has all the 195 countries that we can filter

195

the number of unique countries before filtering:

In [97]:
ei.country.nunique(), hi.country.nunique(), si.country.nunique(), pi.country.nunique()

(227, 262, 226, 166)

keeping only the standard countries

In [98]:
ei = ei[ei.country.isin(countries_to_keep)]
hi = hi[hi.country.isin(countries_to_keep)]
si = si[si.country.isin(countries_to_keep)]
pi = pi[pi.country.isin(countries_to_keep)]

number of unique countries after filtering:

In [99]:
ei.country.nunique(), hi.country.nunique(), si.country.nunique(), pi.country.nunique()

(195, 195, 183, 150)

looking for countries whose region is null

In [100]:
# economic_indicators_df
ei[ei.region.isnull()].country.unique()

<StringArray>
[         'Cuba',       'Eritrea', 'Liechtenstein',        'Monaco',
   'North Korea',   'South Sudan',         'Syria',     'Venezuela',
         'Yemen']
Length: 9, dtype: str

In [101]:
# health_indicators_df
print(hi[hi.region.isnull()].country.unique())
pd.Series(hi[hi.region.isnull()].country.unique()).to_csv('hi_missing_regions.csv')   # exporting the series to see full data, as it is above 120 rows.

<StringArray>
[         'Afghanistan',              'Andorra',               'Angola',
  'Antigua and Barbuda',              'Bahamas',              'Bahrain',
           'Bangladesh',              'Belarus',              'Belgium',
               'Belize',
 ...
              'Ukraine', 'United Arab Emirates',       'United Kingdom',
           'Uzbekistan',              'Vanuatu',            'Venezuela',
              'Vietnam',                'Yemen',               'Zambia',
             'Zimbabwe']
Length: 127, dtype: str


In [102]:
# social_indicators_df
si[si.region.isnull()].country.unique()

<StringArray>
[          'Belize',         'Kiribati', 'Marshall Islands',
            'Nauru',            'Palau',        'Palestine',
            'Samoa',       'San Marino',            'Tonga',
           'Tuvalu']
Length: 10, dtype: str

In [103]:
# country-region mapping dictionary
country_to_region = {
    "Afghanistan": "Asia",
    "Albania": "Europe",
    "Algeria": "Africa",
    "Andorra": "Europe",
    "Angola": "Africa",
    "Antigua and Barbuda": "North America",
    "Argentina": "South America",
    "Armenia": "Asia",
    "Australia": "Oceania",
    "Austria": "Europe",
    "Azerbaijan": "Asia",
    "Bahamas": "North America",
    "Bahrain": "Asia",
    "Bangladesh": "Asia",
    "Barbados": "North America",
    "Belarus": "Europe",
    "Belgium": "Europe",
    "Belize": "North America",
    "Benin": "Africa",
    "Bhutan": "Asia",
    "Bolivia": "South America",
    "Bosnia and Herzegovina": "Europe",
    "Botswana": "Africa",
    "Brazil": "South America",
    "Brunei": "Asia",
    "Bulgaria": "Europe",
    "Burkina Faso": "Africa",
    "Burundi": "Africa",
    "Cambodia": "Asia",
    "Cameroon": "Africa",
    "Canada": "North America",
    "Cape Verde": "Africa",
    "Central African Republic": "Africa",
    "Chad": "Africa",
    "Chile": "South America",
    "China": "Asia",
    "Colombia": "South America",
    "Comoros": "Africa",
    "Congo": "Africa",
    "Costa Rica": "North America",
    "Cote d'Ivoire": "Africa",
    "Croatia": "Europe",
    "Cuba": "North America",
    "Cyprus": "Asia",
    "Czechia": "Europe",
    "Democratic Republic of Congo": "Africa",
    "Denmark": "Europe",
    "Djibouti": "Africa",
    "Dominica": "North America",
    "Dominican Republic": "North America",
    "East Timor": "Asia",
    "Ecuador": "South America",
    "Egypt": "Africa",
    "El Salvador": "North America",
    "Equatorial Guinea": "Africa",
    "Eritrea": "Africa",
    "Estonia": "Europe",
    "Eswatini": "Africa",
    "Ethiopia": "Africa",
    "Fiji": "Oceania",
    "Finland": "Europe",
    "France": "Europe",
    "Gabon": "Africa",
    "Gambia": "Africa",
    "Georgia": "Asia",
    "Germany": "Europe",
    "Ghana": "Africa",
    "Greece": "Europe",
    "Grenada": "North America",
    "Guatemala": "North America",
    "Guinea": "Africa",
    "Guinea-Bissau": "Africa",
    "Guyana": "South America",
    "Haiti": "North America",
    "Honduras": "North America",
    "Hungary": "Europe",
    "Iceland": "Europe",
    "India": "Asia",
    "Indonesia": "Asia",
    "Iran": "Asia",
    "Iraq": "Asia",
    "Ireland": "Europe",
    "Israel": "Asia",
    "Italy": "Europe",
    "Jamaica": "North America",
    "Japan": "Asia",
    "Jordan": "Asia",
    "Kazakhstan": "Asia",
    "Kenya": "Africa",
    "Kiribati": "Oceania",
    "Kosovo": "Europe",
    "Kuwait": "Asia",
    "Kyrgyzstan": "Asia",
    "Laos": "Asia",
    "Latvia": "Europe",
    "Lebanon": "Asia",
    "Lesotho": "Africa",
    "Liberia": "Africa",
    "Libya": "Africa",
    "Liechtenstein": "Europe",
    "Lithuania": "Europe",
    "Luxembourg": "Europe",
    "Madagascar": "Africa",
    "Malawi": "Africa",
    "Malaysia": "Asia",
    "Maldives": "Asia",
    "Mali": "Africa",
    "Malta": "Europe",
    "Marshall Islands": "Oceania",
    "Mauritania": "Africa",
    "Mauritius": "Africa",
    "Mexico": "North America",
    "Micronesia (country)": "Oceania",
    "Moldova": "Europe",
    "Monaco": "Europe",
    "Mongolia": "Asia",
    "Montenegro": "Europe",
    "Morocco": "Africa",
    "Mozambique": "Africa",
    "Myanmar": "Asia",
    "Namibia": "Africa",
    "Nauru": "Oceania",
    "Nepal": "Asia",
    "Netherlands": "Europe",
    "New Zealand": "Oceania",
    "Nicaragua": "North America",
    "Niger": "Africa",
    "Nigeria": "Africa",
    "North Korea": "Asia",
    "North Macedonia": "Europe",
    "Norway": "Europe",
    "Oman": "Asia",
    "Pakistan": "Asia",
    "Palau": "Oceania",
    "Palestine": "Asia",
    "Panama": "North America",
    "Papua New Guinea": "Oceania",
    "Paraguay": "South America",
    "Peru": "South America",
    "Philippines": "Asia",
    "Poland": "Europe",
    "Portugal": "Europe",
    "Qatar": "Asia",
    "Romania": "Europe",
    "Russia": "Europe",
    "Rwanda": "Africa",
    "Saint Kitts and Nevis": "North America",
    "Saint Lucia": "North America",
    "Saint Vincent and the Grenadines": "North America",
    "Samoa": "Oceania",
    "San Marino": "Europe",
    "Sao Tome and Principe": "Africa",
    "Saudi Arabia": "Asia",
    "Senegal": "Africa",
    "Serbia": "Europe",
    "Seychelles": "Africa",
    "Sierra Leone": "Africa",
    "Singapore": "Asia",
    "Slovakia": "Europe",
    "Slovenia": "Europe",
    "Solomon Islands": "Oceania",
    "Somalia": "Africa",
    "South Africa": "Africa",
    "South Korea": "Asia",
    "South Sudan": "Africa",
    "Spain": "Europe",
    "Sri Lanka": "Asia",
    "Sudan": "Africa",
    "Suriname": "South America",
    "Sweden": "Europe",
    "Switzerland": "Europe",
    "Syria": "Asia",
    "Tajikistan": "Asia",
    "Tanzania": "Africa",
    "Thailand": "Asia",
    "Togo": "Africa",
    "Tonga": "Oceania",
    "Trinidad and Tobago": "North America",
    "Tunisia": "Africa",
    "Turkey": "Asia",
    "Turkmenistan": "Asia",
    "Tuvalu": "Oceania",
    "Uganda": "Africa",
    "Ukraine": "Europe",
    "United Arab Emirates": "Asia",
    "United Kingdom": "Europe",
    "United States": "North America",
    "Uruguay": "South America",
    "Uzbekistan": "Asia",
    "Vanuatu": "Oceania",
    "Venezuela": "South America",
    "Vietnam": "Asia",
    "Yemen": "Asia",
    "Zambia": "Africa",
    "Zimbabwe": "Africa"
}

# poverty_indicators_df     (actually this dataset does not have region column (also not code), so we have to add it)
pi['region'] = pi['country'].map(country_to_region)     # adding region column

# rearranging the columns
pi = pi[['country', 'region', 'year', 'poverty_share', 'poverty_number']]

using the region mapping dict, filling the missing regions for each country in all datasets

In [104]:
ei['region'] = ei['region'].fillna(ei['country'].map(country_to_region))
hi['region'] = hi['region'].fillna(hi['country'].map(country_to_region))
si['region'] = si['region'].fillna(si['country'].map(country_to_region))
pi['region'] = pi['region'].fillna(pi['country'].map(country_to_region))

the unique countries dataset, it will be used for creating dimension (country)

In [105]:
# selecting useful columns
unique_countries = ei[['country', 'region']].drop_duplicates(subset=['country']).reset_index(drop=True)

# standardizing the indexes
unique_countries.index = unique_countries.index + 1

os.makedirs(os.path.join(parent_dir, 'processed datasets'), exist_ok=True)
output_folder = Path(parent_dir, "processed datasets")
unique_countries.to_csv(output_folder / 'unique_countries.csv', index=True)

In [ ]:
unique_countries_indexed = unique_countries.reset_index() # adding index as a separate column so that it can be used

# adding indexes to all the datasets for each country, a unique index (id) is set so that it can be used later for data loading in SQL
ei = pd.merge(ei, unique_countries_indexed, on=['country'], how='left')
hi = pd.merge(hi, unique_countries_indexed, on=['country'], how='left')
si = pd.merge(si, unique_countries_indexed, on=['country'], how='left')
pi = pd.merge(pi, unique_countries_indexed, on=['country'], how='left')

selecting the necessary columns

In [115]:
ei = ei[['index', 'year', 'gdp', 'gdp_pc', 'unemployment_rate']]
hi = hi[['index', 'year', 'total_population', 'life_expectancy', 'mortality_rate_u15', 'homicide_rate_p100k']]
si = si[['index', 'year', 'literacy_rate', 'human_rights_index']]
pi = pi[['index', 'year', 'poverty_share', 'poverty_number']]

In [126]:
# fixing datatypes to avoid errors when loading data in sql database

hi['year'] = hi['year'].astype('Int64')
hi['total_population'] = hi['total_population'].astype('Int64')
hi.info()

<class 'pandas.DataFrame'>
RangeIndex: 1755 entries, 0 to 1754
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                1755 non-null   int64  
 1   year                 1755 non-null   Int64  
 2   total_population     1755 non-null   Int64  
 3   life_expectancy      1755 non-null   float64
 4   mortality_rate_u15   1746 non-null   float64
 5   homicide_rate_p100k  1027 non-null   float64
dtypes: Int64(2), float64(3), int64(1)
memory usage: 85.8 KB


saving the datasets

In [127]:
ei.to_csv(output_folder / 'economic_indicators.csv', index=False)
hi.to_csv(output_folder / 'health_indicators.csv', index=False)
si.to_csv(output_folder / 'social_indicators.csv', index=False)
pi.to_csv(output_folder / 'poverty_indicators.csv', index=False)